<a href="https://colab.research.google.com/github/nikolas-joyce/volatility-signals/blob/main/notebooks/alpha_volatility_breakout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alpha: Volatility Breakout + VIX Spike Filter

> Combines two complementary signals. The **volatility breakout** enters long when price exceeds a rolling mean by an adaptive ATR multiple — confirming momentum in expanding markets. The **VIX spike filter** suppresses entries (and closes positions) when VIX crosses above its 5-year 98th-percentile rolling threshold, exiting during regime-level fear events. Together they ride trending, low-volatility environments while stepping aside during market dislocations.

## Notebook structure
| Section | Description |
|---------|-------------|
| 0 | Install & imports |
| 1 | Config & data |
| 2 | Signal functions |
| 3 | Signal generation |
| 4 | Returns & performance |
| 5 | Per-ticker breakdown |
| 6 | Parameter sensitivity |
| 7 | Export signals for swarm |

**Run all cells top-to-bottom in a fresh Colab runtime.**


## Section 0 — Install & Imports

In [ ]:
!pip install yfinance --quiet
print("Dependencies installed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')

try:
    import yfinance as yf
except ImportError as e:
    raise ImportError('yfinance missing — restart runtime after install') from e

print('Imports complete')


## Section 1 — Config & Data

In [ ]:
TICKERS = ['SPY', 'QQQ', 'IWM', 'TLT', 'GLD', 'USO', 'EEM', 'XLF', 'XLE', 'XLK']
START_DATE = '2010-01-01'
COST_BPS   = 5

# Volatility breakout params
VOL_WINDOW      = 120     # ATR and rolling-mean lookback (bars)
VOL_MULTIPLIER  = 0.370   # adaptive ATR multiplier for breakout threshold
MOM_WINDOW      = 120     # momentum confirmation lookback (must equal VOL_WINDOW)

# VIX spike filter params
VIX_ROLL_WINDOW  = 1260   # 5-year rolling window for VIX quantile
VIX_UPPER_Q      = 0.98   # quantile threshold — VIX above this = sell signal
VIX_HOLD_BARS    = 40     # bars to hold the sell signal once triggered

print('Fetching OHLCV...')
raw = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)
assert isinstance(raw.columns, pd.MultiIndex)
close   = raw['Close'].dropna(how='all').ffill(limit=3)
high    = raw['High'].reindex(close.index).ffill(limit=3)
low     = raw['Low'].reindex(close.index).ffill(limit=3)
returns = close.pct_change()

# VIX — fetched separately (single-ticker)
vix_raw = yf.download('^VIX', start='2000-01-01', auto_adjust=True, progress=False)
vix     = vix_raw['Close'].squeeze().rename('^VIX')

keep = close.isnull().mean() < 0.20
close, high, low, returns = (df[keep[keep].index] for df in [close, high, low, returns])
TICKERS = close.columns.tolist()

print(f'  Shape  : {close.shape}')
print(f'  Range  : {close.index[0].date()} -> {close.index[-1].date()}')
print(f'  Tickers: {TICKERS}')
print(f'  VIX    : {vix.index[0].date()} -> {vix.index[-1].date()}')
print('Data ready')


## Section 2 — Helper Functions

In [ ]:
def compute_atr(high, low, close, period=14):
    prev  = close.shift(1)
    tr    = pd.DataFrame(
        np.maximum(np.maximum(high.values - low.values,
                              np.abs(high.values - prev.values)),
                   np.abs(low.values  - prev.values)),
        index=close.index, columns=close.columns)
    return tr.rolling(period, min_periods=period).mean()


def apply_costs(returns, position, cost_bps=5):
    cost     = cost_bps / 10_000
    position = position.reindex(returns.index).fillna(0)
    turnover = position.diff().abs()
    return returns * position - turnover * cost


def performance_summary(r, name=''):
    r = r.dropna()
    if len(r) < 21:
        return pd.Series(dict(zip(
            ['Ann. Return','Ann. Vol','Sharpe','Max DD','Calmar','Hit Rate'],
            [float('nan')]*6)), name=name)
    ann     = 252
    cum     = (1 + r).cumprod()
    ann_ret = (1 + r.mean()) ** ann - 1
    ann_vol = r.std() * np.sqrt(ann)
    sharpe  = ann_ret / ann_vol if ann_vol > 1e-9 else float('nan')
    mdd     = (cum / cum.cummax() - 1).min()
    calmar  = ann_ret / abs(mdd) if abs(mdd) > 1e-9 else float('nan')
    hits    = (r > 0).mean()
    return pd.Series({
        'Ann. Return': ann_ret, 'Ann. Vol': ann_vol,
        'Sharpe': sharpe, 'Max DD': mdd,
        'Calmar': calmar, 'Hit Rate': hits,
    }, name=name)

print('Helpers defined')


## Section 3 — Signal Functions

In [ ]:
def volatility_breakout_long(close, high, low, vol_window=120, multiplier=0.370):
    """
    Long entry when close > rolling_mean + adaptive_ATR_multiple.
    Confirmed by positive momentum over the same window.
    ATR uses correct true range: max(H-L, |H-C[-1]|, |L-C[-1]|).
    """
    atr        = compute_atr(high, low, close, period=vol_window)
    vol_scale  = atr.rolling(vol_window).quantile(0.9)
    adapt_mult = multiplier * (atr / vol_scale).fillna(1)
    roll_mean  = close.rolling(vol_window).mean()
    momentum   = close.pct_change(vol_window)
    raw        = close > (roll_mean + adapt_mult)
    confirmed  = raw & (momentum > 0)
    return confirmed.shift(1).fillna(False).astype(int)


def vix_sell_signal(vix, close_index,
                    roll_window=1260, upper_q=0.98, hold_bars=40):
    """
    Sell / avoid signal when VIX crosses above its rolling upper quantile.
    Signal stays active for hold_bars bars after the trigger.
    Returns a boolean Series aligned to close_index.
    """
    vix_aligned = vix.reindex(close_index).ffill()
    roll_upper  = vix_aligned.rolling(roll_window, min_periods=252).quantile(upper_q)
    trigger     = (vix_aligned > roll_upper).astype(int)
    # Extend trigger forward for hold_bars
    sell        = trigger.rolling(hold_bars, min_periods=1).max().astype(bool)
    return sell

print('Signal functions defined')


## Section 4 — Signal Generation

In [ ]:
# Compute signals
long_raw = volatility_breakout_long(
    close, high, low,
    vol_window=VOL_WINDOW,
    multiplier=VOL_MULTIPLIER,
)

sell_mask = vix_sell_signal(
    vix, close.index,
    roll_window=VIX_ROLL_WINDOW,
    upper_q=VIX_UPPER_Q,
    hold_bars=VIX_HOLD_BARS,
)

# Apply VIX filter: zero out long positions during sell-signal periods
v_long  = long_raw.multiply(~sell_mask, axis=0).astype(int)
v_short = pd.DataFrame(0, index=close.index, columns=close.columns)  # long-only strategy

print(f'Breakout long density (pre-VIX filter) : {long_raw.mean().mean():.1%}')
print(f'VIX sell-signal density                : {sell_mask.mean():.1%}')
print(f'Net long density (post-VIX filter)     : {v_long.mean().mean():.1%}')

# Quick equity curve preview
pos_series = v_long.mean(axis=1)  # equal-weight position across tickers
pnl = (returns.mean(axis=1) * pos_series).dropna()
cum = (1 + pnl).cumprod()
fig, ax = plt.subplots(figsize=(14, 4))
cum.plot(ax=ax, title='Volatility Breakout + VIX Filter — Equally-Weighted Preview')
ax.set_ylabel('Cumulative return')
plt.tight_layout()
plt.show()


## Section 5 — Returns & Performance

In [ ]:
def compute_alpha_returns(long_sig, short_sig, returns, label, cost_bps=COST_BPS):
    def _side(sig, direction, side_label):
        pos = sig.reindex(returns.index).fillna(0) * direction
        net = apply_costs(returns, pos, cost_bps)
        n   = sig.sum(axis=1).replace(0, float('nan'))
        pnl = net.sum(axis=1)
        return (pnl / n).fillna(0).rename(f'{label}_{side_label}')
    l_ret    = _side(long_sig,  +1, 'Long')
    s_ret    = _side(short_sig, -1, 'Short') if short_sig.any().any() else pd.Series(0, index=l_ret.index, name=f'{label}_Short')
    combined = ((l_ret + s_ret) / 2).rename(label)
    return l_ret, s_ret, combined


a_long_ret, _, a_combined = compute_alpha_returns(v_long, v_short, returns, 'Vol_Breakout_VIX')

# Benchmark: buy-and-hold SPY
spy_ret = returns['SPY'].dropna() if 'SPY' in returns.columns else returns.iloc[:, 0].dropna()

for r, name in [(a_long_ret, 'Vol Breakout (Long)'), (a_combined, 'Combined'), (spy_ret, 'SPY B&H')]:
    s = performance_summary(r, name=name)
    print(s.round(3).to_string())
    print()

# Equity curve comparison
cum_alpha = (1 + a_long_ret.dropna()).cumprod()
cum_spy   = (1 + spy_ret.reindex(cum_alpha.index).fillna(0)).cumprod()
fig, ax   = plt.subplots(figsize=(14, 5))
cum_alpha.plot(ax=ax, label='Vol Breakout + VIX Filter')
cum_spy.plot(ax=ax, label='SPY Buy-and-Hold', linestyle='--')
ax.set_title('Volatility Breakout Alpha — Equity Curve vs SPY')
ax.set_ylabel('Cumulative return')
ax.legend()
plt.tight_layout()
plt.show()


## Section 6 — Per-Ticker Breakdown

In [ ]:
def per_ticker_sharpe(long_sig, returns, label):
    sharpes = {}
    for ticker in TICKERS:
        lp  = long_sig[ticker].reindex(returns.index).fillna(0)
        net = apply_costs(returns[[ticker]], lp.to_frame(), COST_BPS)[ticker]
        std = net.std()
        sharpes[ticker] = (net.mean() / std * np.sqrt(252) if std > 1e-9 else float('nan'))
    return pd.Series(sharpes, name=label)

ts = per_ticker_sharpe(v_long, returns, 'Vol_Breakout_VIX')
print(f'Per-Ticker Annualised Sharpe:')
print(ts.sort_values(ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 4))
ts.sort_values().plot(kind='barh', ax=ax, title='Volatility Breakout — Per-Ticker Sharpe')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()


## Section 7 — Parameter Sensitivity

In [ ]:
import itertools

sweep_rows = []
for vw, mult, vq in itertools.product([60, 90, 120, 180], [0.25, 0.37, 0.50], [0.95, 0.98]):
    lg  = volatility_breakout_long(close, high, low, vol_window=vw, multiplier=mult)
    sm  = vix_sell_signal(vix, close.index, roll_window=VIX_ROLL_WINDOW,
                          upper_q=vq, hold_bars=VIX_HOLD_BARS)
    vl  = lg.multiply(~sm, axis=0).astype(int)
    lr, _, _ = compute_alpha_returns(vl, v_short, returns, f'vw{vw}_m{mult}_vq{vq}')
    sweep_rows.append({
        **performance_summary(lr).to_dict(),
        'vol_window': vw, 'multiplier': mult, 'vix_q': vq,
    })

sweep_df = pd.DataFrame(sweep_rows).sort_values('Sharpe', ascending=False)
print('Top 10 parameter combinations by Sharpe:')
print(sweep_df[['vol_window','multiplier','vix_q','Sharpe','Ann. Return','Max DD']].head(10).round(3).to_string())


## Section 8 — Export Signals for Swarm

In [ ]:
import os, pickle
os.makedirs('exports', exist_ok=True)
with open('exports/alpha_volatility_breakout.pkl', 'wb') as f:
    pickle.dump({'long': v_long, 'short': v_short}, f)
print('Signals exported -> exports/alpha_volatility_breakout.pkl')
